# Creating a Semantic Relationships Graph from an IFC file

In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed TopologicPy Classes

In [2]:
from topologicpy.Topology import Topology
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Color import Color
from topologicpy.IFC import IFC
from topologicpy.CellComplex import CellComplex

## 2. Check the TopologicPy version

In [3]:
print("This tutorial requires topologicpy version 0.9.30 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.30 or newer.
The version that you are using (0.9.30) is NEWER than the latest version (0.9.29) available from PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 5. Specify the path to the IFC file and what to include

In [5]:
# Choose your own IFC file if you wish
#ifc_file_path = r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\NVW_DCR-LOD100_Arch.ifc"
ifc_file_path = r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\NVW_DCR-LOD200_Arch.ifc"

ifc_file = IFC.FileByPath(ifc_file_path)

IFCFastTopology.Parse - Parsed 515000 entities in 3.760s.


## 6. List all the IFC element types found in the IFC file

In [6]:
element_types = IFC.ElementTypes(ifc_file, includeCounts = False, silent = False)
element_types = [e.lower() for e in element_types]
for element_type in element_types:
    print(element_type)

ifcapplication
ifcarbitraryclosedprofiledef
ifcarbitraryprofiledefwithvoids
ifcaxis2placement2d
ifcaxis2placement3d
ifcbooleanclippingresult
ifcboundingbox
ifcbuilding
ifcbuildingelementproxy
ifcbuildingstorey
ifccartesianpoint
ifccartesiantransformationoperator3d
ifcclassificationreference
ifccolumn
ifcconnectionsurfacegeometry
ifcconversionbasedunit
ifccurveboundedplane
ifcdimensionalexponents
ifcdirection
ifcdoor
ifcdoorstyle
ifcelementquantity
ifcextrudedareasolid
ifcface
ifcfaceouterbound
ifcgeometricrepresentationcontext
ifclocalplacement
ifcmappeditem
ifcmaterial
ifcmateriallayer
ifcmateriallayerset
ifcmateriallayersetusage
ifcmateriallist
ifcmeasurewithunit
ifcmember
ifcopeningelement
ifcopenshell
ifcorganization
ifcownerhistory
ifcperson
ifcpersonandorganization
ifcplane
ifcpolygonalboundedhalfspace
ifcpolyline
ifcpolyloop
ifcpostaladdress
ifcproductdefinitionshape
ifcproject
ifcpropertyset
ifcpropertysinglevalue
ifcquantityarea
ifcquantitylength
ifcrailing
ifcramp
ifcrectangl

## 7. List all the relationship types found in the IFC file

In [7]:
rel_types = IFC.RelationshipTypes(ifc_file, includeCounts = False, silent = False)
rel_types = [r.lower() for r in rel_types]
for rel_type in rel_types:
    print(rel_type)

ifcrelaggregates
ifcrelassociatesclassification
ifcrelassociatesmaterial
ifcrelconnectspathelements
ifcrelcontainedinspatialstructure
ifcreldefinesbyproperties
ifcreldefinesbytype
ifcrelfillselement
ifcrelspaceboundary
ifcrelvoidselement


## 8. Convert the IFC file to a graph

In [8]:
from time import time
start = time()
graph = Graph.ByIFCPath(ifc_file_path,
                        importMode="topology",
                        dictionaryMode="all",
                        xMin = -5, yMin = -5, zMin = -5,
                        xMax = 5, yMax = 5, zMax = 5,
                        colorScale="twilight"
                       )
end = time()
print("Duration:", round(end-start, 2))

Duration: 13.01


## 10. Query and mutate the IFC graph using the experimental GQL engine

The following cells exercise the new GQL-inspired query capabilities on the IFC-derived TopologicPy graph created above. They are deliberately written to adapt to the actual IFC element and relationship dictionaries found in the file.

In [9]:
# Import the experimental GQL wrapper.
# Make sure Parser.py, Executor.py, and GQL.py are in topologicpy/GQL/ before running this cell.
from topologicpy.GQL import GQL
from topologicpy.gql.Parser import Parser
from topologicpy.gql.Executor import Executor

from collections import Counter
import re

try:
    import pandas as pd
except Exception:
    pd = None

print("GQL imports completed.")


GQL imports completed.


In [10]:
# Helper functions for inspecting and normalising dictionaries.
# The GQL executor currently matches node and edge labels using the dictionary keys "label" or "category".
# IFC-derived graphs often use keys such as "IFC_type" instead, so we add safe GQL categories while preserving existing dictionary data.

def _dict_to_python(value):
    """
    Convert Topologic objects to their dictionaries for display.
    Leave scalars unchanged.
    """

    if value is None:
        return None

    if isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, (list, tuple)):
        return type(value)(_dict_to_python(v) for v in value)

    if isinstance(value, dict):
        return {
            k: _dict_to_python(v)
            for k, v in value.items()
        }

    from contextlib import redirect_stdout
    import io

    from topologicpy.Topology import Topology
    from topologicpy.Dictionary import Dictionary

    try:
        if not Topology.IsInstance(value, "topology"):
            return repr(value)
    except Exception:
        return repr(value)

    try:
        buffer = io.StringIO()
        with redirect_stdout(buffer):
            d = Topology.Dictionary(value)
    except Exception:
        return repr(value)

    if d is None:
        return repr(value)

    try:
        if hasattr(Dictionary, "PythonDictionary"):
            result = Dictionary.PythonDictionary(d)
            return result if isinstance(result, dict) else repr(value)

        if hasattr(Dictionary, "Keys"):
            keys = Dictionary.Keys(d)
            return {
                key: Dictionary.ValueAtKey(d, key, None)
                for key in keys
            }
    except Exception:
        return repr(value)

    return repr(value)


def _set_python_dict(topology, data):
    """Set a Python dictionary on a topology and return the topology."""
    keys = list(data.keys())
    values = [data[k] for k in keys]
    return Topology.SetDictionary(topology, Dictionary.ByKeysValues(keys, values))


def _value(topology, key, default=None):
    try:
        return Dictionary.ValueAtKey(Topology.Dictionary(topology), key, default)
    except Exception:
        return default


def _first_value(data, keys, default=None):
    for key in keys:
        value = data.get(key, None)
        if value not in (None, ""):
            return value
    return default


def _safe_gql_name(value, default="Item"):
    """Convert arbitrary IFC labels to parser-safe names."""
    if value is None:
        value = default
    value = str(value).strip()
    value = re.sub(r"[^A-Za-z0-9_]", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    if not value:
        value = default
    if value[0].isdigit():
        value = "_" + value
    return value


def _summarise_value(value):
    """Make query results readable when they contain Topologic objects."""
    try:
        data = _dict_to_python(value)
        name = _first_value(data, ["name", "Name", "LongName", "GlobalId", "id", "uuid"], None)
        category = _first_value(data, ["category", "label", "IFC_type", "ifc_type", "type"], None)
        if name or category:
            return f"{category or type(value).__name__}:{name or ''}"
    except Exception:
        pass
    return value


def display_rows(rows, max_rows=20):
    """Display GQL rows as a compact table when pandas is available."""
    if isinstance(rows, dict) and "rows" in rows:
        rows = rows["rows"]

    if rows is None:
        print("No rows returned.")
        return

    readable = []
    for row in rows[:max_rows]:
        readable.append({k: _summarise_value(v) for k, v in row.items()})

    if pd is not None:
        display(pd.DataFrame(readable))
    else:
        for row in readable:
            print(row)

    if len(rows) > max_rows:
        print(f"Showing {max_rows} of {len(rows)} rows.")


In [11]:
# Prepare a working graph for GQL experiments.
# This does not prevent you from keeping the original `graph`; mutation examples below use `gql_graph`.

vertices = Graph.Vertices(graph)
edges = Graph.Edges(graph)

vertex_type_keys = ["category", "label", "IFC_type", "ifc_type", "IfcType", "type", "class", "Class"]
vertex_name_keys = ["name", "Name", "LongName", "GlobalId", "id", "uuid"]
edge_type_keys = ["category", "label", "IFC_type", "ifc_type", "IfcType", "type", "relationship_type", "RelationshipType", "rel_type", "RelType"]

for v in vertices:
    data = _dict_to_python(v)
    raw_type = _first_value(data, vertex_type_keys, "Node")
    raw_name = _first_value(data, vertex_name_keys, None)
    data.setdefault("raw_category", raw_type)
    data["category"] = _safe_gql_name(raw_type, "Node")
    if raw_name is not None:
        data.setdefault("name", str(raw_name))
    _set_python_dict(v, data)

for e in edges:
    data = _dict_to_python(e)
    raw_type = _first_value(data, edge_type_keys, "RELATES")
    data.setdefault("raw_category", raw_type)
    data["category"] = _safe_gql_name(raw_type, "RELATES")
    _set_python_dict(e, data)

vertex_categories = Counter(_value(v, "category", "Node") for v in vertices)
edge_categories = Counter(_value(e, "category", "RELATES") for e in edges)

print("Vertices:", len(vertices))
print("Edges:", len(edges))
print("Most common vertex categories:")
for item, count in vertex_categories.most_common(10):
    print(f"  {item}: {count}")
print("Most common edge categories:")
for item, count in edge_categories.most_common(10):
    print(f"  {item}: {count}")

primary_node_category = vertex_categories.most_common(1)[0][0] if vertex_categories else "Node"
primary_edge_category = edge_categories.most_common(1)[0][0] if edge_categories else "RELATES"
print("Primary node category for examples:", primary_node_category)
print("Primary edge category for examples:", primary_edge_category)


Vertices: 4982
Edges: 7783
Most common vertex categories:
  IfcPropertySet: 1258
  IfcOpeningElement: 747
  IfcMaterialLayerSetUsage: 590
  IfcWallStandardCase: 501
  IfcMember: 450
  IfcWindow: 291
  IfcWindowStyle: 291
  IfcColumn: 284
  IfcSlab: 119
  IfcSpace: 89
Most common edge categories:
  IfcRelAssociatesMaterial: 1763
  IfcRelContainedInSpatialStructure: 1685
  IfcRelDefinesByProperties: 1266
  IfcRelSpaceBoundary: 1055
  IfcRelVoidsElement: 747
  IfcRelConnectsPathElements: 345
  IfcRelFillsElement: 315
  IfcRelDefinesByType: 315
  IfcRelAggregates: 203
  IfcRelAssociatesClassification: 89
Primary node category for examples: IfcPropertySet
Primary edge category for examples: IfcRelAssociatesMaterial


### 10.1 Read queries

These cells exercise `MATCH`, `WHERE`, `RETURN`, aliases, `DISTINCT`, `COUNT`, `ORDER BY`, `SKIP`, and `LIMIT`.

In [12]:
from time import time

from time import perf_counter


query = """
MATCH (a)--(b)
RETURN COUNT(*) AS pair_count
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("First run:", perf_counter() - t0, rows)

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("Second run:", perf_counter() - t0, rows)


from time import time
# Query 1: Count all undirected adjacency pairs in the IFC graph.
query = """
MATCH (a)--(b)
RETURN COUNT(*) AS pair_count
"""

start = time()
rows = GQL.Query(graph, query)
end = time()
print("MATCH Duration:", round(end-start, 2))
display_rows(rows)


First run: 9.852484700037166 [{'pair_count': 15566}]
Second run: 0.0003066000062972307 [{'pair_count': 15566}]
MATCH Duration: 0.0


,pair_count
0,15566


In [13]:
# Query 2: List the distinct IFC-derived node categories.

query = """
MATCH (a)--(b)
RETURN DISTINCT a.category AS node_type
ORDER BY node_type
LIMIT 25
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
end = time()
print("Query run:", round(perf_counter() - t0, 2))
display_rows(rows)


Query run: 0.52


,node_type
0,IfcBuilding
1,IfcBuildingElementProxy
2,IfcBuildingStorey
3,IfcClassificationReference
4,IfcColumn
5,IfcDoor
6,IfcDoorStyle
7,IfcElementQuantity
8,IfcMaterial
9,IfcMaterialLayerSetUsage


Showing 20 of 25 rows.


In [14]:
# Query 3: List the distinct IFC-derived relationship categories.
query = """
MATCH (a)-[r]->(b)
RETURN DISTINCT r.category AS relationship_type
ORDER BY relationship_type
LIMIT 25
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("Query run:", round(perf_counter() - t0, 2))
display_rows(rows)


Query run: 0.37


,relationship_type
0,IfcRelAggregates
1,IfcRelAssociatesClassification
2,IfcRelAssociatesMaterial
3,IfcRelConnectsPathElements
4,IfcRelContainedInSpatialStructure
5,IfcRelDefinesByProperties
6,IfcRelDefinesByType
7,IfcRelFillsElement
8,IfcRelSpaceBoundary
9,IfcRelVoidsElement


In [15]:
# Query 4: Directed relationship sample with aliases and ordering.
query = """
MATCH (a)-[r]->(b)
RETURN a.category AS source_type, r.category AS relationship, b.category AS target_type
ORDER BY source_type ASC, relationship ASC, target_type ASC
LIMIT 25
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("Query run:", round(perf_counter() - t0, 2))
display_rows(rows)


Query run: 0.65


,source_type,relationship,target_type
0,IfcBuilding,IfcRelAggregates,IfcBuildingStorey
1,IfcBuilding,IfcRelAggregates,IfcBuildingStorey
2,IfcBuilding,IfcRelAggregates,IfcBuildingStorey
3,IfcBuilding,IfcRelAggregates,IfcBuildingStorey
4,IfcBuilding,IfcRelAggregates,IfcBuildingStorey
5,IfcBuilding,IfcRelAggregates,IfcBuildingStorey
6,IfcBuilding,IfcRelAggregates,IfcBuildingStorey
7,IfcBuilding,IfcRelAggregates,IfcBuildingStorey
8,IfcBuildingStorey,IfcRelAggregates,IfcSpace
9,IfcBuildingStorey,IfcRelAggregates,IfcSpace


Showing 20 of 25 rows.


In [16]:
# Query 5: Filter by the most common node category discovered above.
query = f"""
MATCH (a:{primary_node_category})-[r]->(b)
RETURN a.category AS source_type, a.name AS source_name, r.category AS relationship, b.category AS target_type, b.name AS target_name
ORDER BY source_type ASC, target_type ASC
LIMIT 25
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("Query run:", round(perf_counter() - t0, 2))
display_rows(rows)


Query run: 0.51


,source_type,source_name,relationship,target_type,target_name
0,IfcPropertySet,None,IfcRelDefinesByProperties,IfcBuilding,None
1,IfcPropertySet,None,IfcRelDefinesByProperties,IfcBuildingStorey,None
2,IfcPropertySet,None,IfcRelDefinesByProperties,IfcBuildingStorey,None
3,IfcPropertySet,None,IfcRelDefinesByProperties,IfcBuildingStorey,None
4,IfcPropertySet,None,IfcRelDefinesByProperties,IfcBuildingStorey,None
5,IfcPropertySet,None,IfcRelDefinesByProperties,IfcBuildingStorey,None
6,IfcPropertySet,None,IfcRelDefinesByProperties,IfcBuildingStorey,None
7,IfcPropertySet,None,IfcRelDefinesByProperties,IfcBuildingStorey,None
8,IfcPropertySet,None,IfcRelDefinesByProperties,IfcBuildingStorey,None
9,IfcPropertySet,None,IfcRelDefinesByProperties,IfcColumn,None


Showing 20 of 25 rows.


In [17]:
# Query 6: Filter by the most common relationship category discovered above.
query = f"""
MATCH (a)-[r:{primary_edge_category}]->(b)
RETURN a.category AS source_type, r.category AS relationship, b.category AS target_type
ORDER BY source_type ASC, target_type ASC
LIMIT 25
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("Query run:", round(perf_counter() - t0, 2))
display_rows(rows)


Query run: 0.36


,source_type,relationship,target_type
0,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy
1,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy
2,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy
3,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy
4,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy
5,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy
6,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy
7,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy
8,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy
9,IfcMaterial,IfcRelAssociatesMaterial,IfcBuildingElementProxy


Showing 20 of 25 rows.


In [18]:
# Query 7: Boolean logic with parentheses.
# This returns rows where either endpoint belongs to the most common node category.
query = f"""
MATCH (a)-[r]->(b)
WHERE (a.category = '{primary_node_category}' OR b.category = '{primary_node_category}')
RETURN a.category AS source_type, r.category AS relationship, b.category AS target_type
ORDER BY source_type ASC, relationship ASC, target_type ASC
LIMIT 25
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("Query run:", round(perf_counter() - t0, 2))
display_rows(rows)

Query run: 0.56


,source_type,relationship,target_type
0,IfcPropertySet,IfcRelDefinesByProperties,IfcBuilding
1,IfcPropertySet,IfcRelDefinesByProperties,IfcBuildingStorey
2,IfcPropertySet,IfcRelDefinesByProperties,IfcBuildingStorey
3,IfcPropertySet,IfcRelDefinesByProperties,IfcBuildingStorey
4,IfcPropertySet,IfcRelDefinesByProperties,IfcBuildingStorey
5,IfcPropertySet,IfcRelDefinesByProperties,IfcBuildingStorey
6,IfcPropertySet,IfcRelDefinesByProperties,IfcBuildingStorey
7,IfcPropertySet,IfcRelDefinesByProperties,IfcBuildingStorey
8,IfcPropertySet,IfcRelDefinesByProperties,IfcBuildingStorey
9,IfcPropertySet,IfcRelDefinesByProperties,IfcColumn


Showing 20 of 25 rows.


In [19]:
# Query 8: Pagination with ORDER BY, SKIP, and LIMIT.
query = """
MATCH (a)--(b)
RETURN a.category AS source_type, b.category AS target_type
ORDER BY source_type ASC, target_type ASC
SKIP 10
LIMIT 10
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("Query run:", round(perf_counter() - t0, 2))
display_rows(rows)


Query run: 0.77


,source_type,target_type
0,IfcBuildingElementProxy,IfcBuildingStorey
1,IfcBuildingElementProxy,IfcBuildingStorey
2,IfcBuildingElementProxy,IfcBuildingStorey
3,IfcBuildingElementProxy,IfcBuildingStorey
4,IfcBuildingElementProxy,IfcBuildingStorey
5,IfcBuildingElementProxy,IfcBuildingStorey
6,IfcBuildingElementProxy,IfcBuildingStorey
7,IfcBuildingElementProxy,IfcBuildingStorey
8,IfcBuildingElementProxy,IfcBuildingStorey
9,IfcBuildingElementProxy,IfcBuildingStorey


### 10.2 Mutation queries

The following cells exercise `CREATE`, `MERGE`, `SET`, and `DELETE` on a working graph variable called `gql_graph`. The original variable `graph` remains available above. If your local mutation grammar differs slightly, the helper will print the parser error and continue.

In [20]:
def run_mutation(title, query, update_graph=True):
    """Run a mutation query and update graph if the result returns an updated graph."""
    global graph
    print("\n" + "=" * 80)
    print(title)
    print(query)

    result = GQL.Query(graph, query)

    if isinstance(result, dict):
        print("Action:", result.get("action"))
        print("Created:", result.get("created"), "Matched:", result.get("matched"), "Updated:", result.get("updated"), "Deleted:", result.get("deleted"))
        if update_graph and result.get("graph", None) is not None:
            graph = result["graph"]
        display_rows(result.get("rows", []))
    else:
        display_rows(result)

    return result


In [21]:
# Mutation 1: CREATE a small synthetic test relationship inside the working graph.
# The synthetic category names are deliberately parser-safe.
query = """
CREATE (a:GQL_Test_Node {name:'Synthetic_A', source:'GQL demo'})-[:GQL_TEST_REL {weight:0.99, source:'GQL demo'}]->(b:GQL_Test_Node {name:'Synthetic_B', source:'GQL demo'})
RETURN a.name AS source, b.name AS target
"""

result = run_mutation("CREATE synthetic test nodes and edge", query)
graph = result['graph']



CREATE synthetic test nodes and edge

CREATE (a:GQL_Test_Node {name:'Synthetic_A', source:'GQL demo'})-[:GQL_TEST_REL {weight:0.99, source:'GQL demo'}]->(b:GQL_Test_Node {name:'Synthetic_B', source:'GQL demo'})
RETURN a.name AS source, b.name AS target

Action: CREATE
Created: 3 Matched: 0 Updated: 0 Deleted: 0


,source,target
0,Synthetic_A,Synthetic_B


In [22]:
# Confirm that the CREATE result is now queryable.
query = """
MATCH (a:GQL_Test_Node)-[r:GQL_TEST_REL]->(b:GQL_Test_Node)
RETURN a.name AS source, r.weight AS weight, b.name AS target
ORDER BY source
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("Query run:", round(perf_counter() - t0, 2))
display_rows(rows)


Query run: 0.22


,source,weight,target
0,Synthetic_A,0.99,Synthetic_B


In [23]:
# Mutation 2: SET a property on the synthetic nodes.
query = """
MATCH (a:GQL_Test_Node)-[r:GQL_TEST_REL]->(b:GQL_Test_Node)
SET a.reviewed = 'yes', b.reviewed = 'yes', r.reviewed = 'yes'
RETURN a.name AS source, a.reviewed AS source_reviewed, r.reviewed AS edge_reviewed, b.name AS target, b.reviewed AS target_reviewed
"""

result = run_mutation("SET properties on synthetic nodes and edge", query)



SET properties on synthetic nodes and edge

MATCH (a:GQL_Test_Node)-[r:GQL_TEST_REL]->(b:GQL_Test_Node)
SET a.reviewed = 'yes', b.reviewed = 'yes', r.reviewed = 'yes'
RETURN a.name AS source, a.reviewed AS source_reviewed, r.reviewed AS edge_reviewed, b.name AS target, b.reviewed AS target_reviewed

Action: SET
Created: 0 Matched: 1 Updated: 3 Deleted: 0


,source,source_reviewed,edge_reviewed,target,target_reviewed
0,Synthetic_A,yes,yes,Synthetic_B,yes


In [24]:
# Mutation 3: MERGE the same synthetic relationship.
# A conservative MERGE should avoid creating unnecessary duplicates when the same pattern already exists.
query = """
MERGE (a:GQL_Test_Node {name:'Synthetic_A'})-[:GQL_TEST_REL {weight:0.99}]->(b:GQL_Test_Node {name:'Synthetic_B'})
RETURN a.name AS source, b.name AS target
"""

result = run_mutation("MERGE synthetic relationship", query)



MERGE synthetic relationship

MERGE (a:GQL_Test_Node {name:'Synthetic_A'})-[:GQL_TEST_REL {weight:0.99}]->(b:GQL_Test_Node {name:'Synthetic_B'})
RETURN a.name AS source, b.name AS target

Action: MERGE
Created: 0 Matched: 1 Updated: 0 Deleted: 0


,source,target
0,Synthetic_A,Synthetic_B


In [25]:
# Confirm the number of synthetic relationships after CREATE + MERGE.
query = """
MATCH (a:GQL_Test_Node)-[r:GQL_TEST_REL]->(b:GQL_Test_Node)
RETURN COUNT(*) AS synthetic_relationship_count
"""

t0 = perf_counter()
rows = GQL.Query(graph, query)
print("Query run:", round(perf_counter() - t0, 2))
display_rows(rows)


Query run: 0.2


,synthetic_relationship_count
0,1


In [26]:
# Mutation 4: DELETE the synthetic test relationship/nodes from the working graph.
# This keeps the IFC-derived graph clean after the mutation demonstration.
query = """
MATCH (a:GQL_Test_Node)-[r:GQL_TEST_REL]->(b:GQL_Test_Node)
DELETE a, r, b
RETURN COUNT(*) AS deleted_pattern_count
"""

t0 = perf_counter()
result = run_mutation("DELETE synthetic test pattern", query)
print("Query run:", round(perf_counter() - t0, 2))
print(result)



DELETE synthetic test pattern

MATCH (a:GQL_Test_Node)-[r:GQL_TEST_REL]->(b:GQL_Test_Node)
DELETE a, r, b
RETURN COUNT(*) AS deleted_pattern_count

Action: DELETE
Created: 0 Matched: 1 Updated: 0 Deleted: 3


,deleted_pattern_count
0,1


Query run: 0.69
{'graph': {'type': 'GQLWorkingGraph', 'vertices': [<topologic_core.Vertex object at 0x0000027F6CA8F230>, <topologic_core.Vertex object at 0x0000027F6CA8E7F0>, <topologic_core.Vertex object at 0x0000027F6CA8F3B0>, <topologic_core.Vertex object at 0x0000027F6CA8F270>, <topologic_core.Vertex object at 0x0000027F6CA8F630>, <topologic_core.Vertex object at 0x0000027F0376DB70>, <topologic_core.Vertex object at 0x0000027F0376ECF0>, <topologic_core.Vertex object at 0x0000027F0376D270>, <topologic_core.Vertex object at 0x0000027F0376D9B0>, <topologic_core.Vertex object at 0x0000027F0376C630>, <topologic_core.Vertex object at 0x0000027F0376E8F0>, <topologic_core.Vertex object at 0x0000027F0376FFB0>, <topologic_core.Vertex object at 0x0000027F0376DEB0>, <topologic_core.Vertex object at 0x0000027F0376F2F0>, <topologic_core.Vertex object at 0x0000027F0376CA30>, <topologic_core.Vertex object at 0x0000027F0376EAB0>, <topologic_core.Vertex object at 0x0000027F0376D130>, <topologic_core

In [27]:
# # Confirm cleanup.
# query = """
# MATCH (a:GQL_Test_Node)--(b:GQL_Test_Node)
# RETURN COUNT(*) AS remaining_synthetic_pairs
# """

# rows = GQL.Query(result['graph'], query)
# display_rows(rows)

query = """
MATCH (a:GQL_Test_Node)-[r:GQL_TEST_REL]->(b:GQL_Test_Node)
RETURN COUNT(*) AS remaining_synthetic_edges
"""

rows = GQL.Query(result['graph'], query)
display_rows(rows)


,remaining_synthetic_edges
0,0


### 10.3 Suggested next experiments

Try replacing `primary_node_category` and `primary_edge_category` with IFC categories that are meaningful in your file, such as spaces, walls, slabs, openings, doors, containment relationships, or decomposition relationships. The same query forms can then be used to inspect IFC semantics at scale.

In [28]:
topologic_graph = GQL.TopologicGraph(result["graph"])
print(topologic_graph)

In [29]:
verts = Graph.Vertices(topologic_graph)
print(len(verts))
for v in verts[:10]:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

4982
['IFC_global_id', 'IFC_id', 'IFC_key', 'IFC_type', 'category', 'color', 'index', 'raw_category'] ['3G_vI0vKzDFwKZy1$6iXQ$', 469755, '3G_vI0vKzDFwKZy1$6iXQ$', 'IfcBuildingElementProxy', 'IfcBuildingElementProxy', '#C9CED9', 0, 'IfcBuildingElementProxy']
['IFC_global_id', 'IFC_id', 'IFC_key', 'IFC_type', 'category', 'color', 'index', 'raw_category'] ['1aHXEPkALF_8JdDgaXeMCR', 469833, '1aHXEPkALF_8JdDgaXeMCR', 'IfcBuildingElementProxy', 'IfcBuildingElementProxy', '#C9CED9', 1, 'IfcBuildingElementProxy']
['IFC_global_id', 'IFC_id', 'IFC_key', 'IFC_type', 'category', 'color', 'index', 'raw_category'] ['3o1MX0t_v5CuUTk7RSULj0', 469911, '3o1MX0t_v5CuUTk7RSULj0', 'IfcBuildingElementProxy', 'IfcBuildingElementProxy', '#C9CED9', 2, 'IfcBuildingElementProxy']
['IFC_global_id', 'IFC_id', 'IFC_key', 'IFC_type', 'category', 'color', 'index', 'raw_category'] ['35crPCLWf4LfF6foWbZ9CS', 469989, '35crPCLWf4LfF6foWbZ9CS', 'IfcBuildingElementProxy', 'IfcBuildingElementProxy', '#C9CED9', 3, 'IfcBuildi